# 04 — Feature Selection

**Primary:** Zhilin Zhang  
**Support:** Tianyi Qin

This notebook uses the same processed rows, train/test split and full feature set as the modelling notebook.

Required outputs:
- one **embedded** method;
- one **filter** method;
- top 3 features from each with scores;
- an explicit comparison of disagreement;
- one concrete hard-case listing ID with actual attributes.

## 1. Imports and data

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

REPO_ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
DATA_PATH = REPO_ROOT / "data" / "processed_listings.csv"
MODEL_SUMMARY_PATH = REPO_ROOT / "output" / "tables" / "model_summary.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError("Run 01_preprocessing.ipynb first.")

if not MODEL_SUMMARY_PATH.exists():
    raise FileNotFoundError("Run 03_modelling.ipynb first.")

df = pd.read_csv(DATA_PATH)
model_summary = pd.read_csv(MODEL_SUMMARY_PATH)

print("Processed shape:", df.shape)
print("Split counts:", df["split"].value_counts().to_dict())

Processed shape: (14472, 20)
Split counts: {'train': 11577, 'test': 2895}


## 2. Reuse the exact modelling feature set and training rows

In [2]:
FEATURES = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "distance_cbd_km",
    "amenity_count",
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_kitchen",
    "has_washer",
    "has_dryer",
]

TARGET = "high_price"
ID_COL = "id"

missing = [c for c in FEATURES + [TARGET, ID_COL, "split"] if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

train_df = df.loc[df["split"].eq("train")].copy()
test_df = df.loc[df["split"].eq("test")].copy()

X_train = train_df[FEATURES]
y_train = train_df[TARGET].astype(int)

print("Training rows:", len(train_df))
print("Target counts:", y_train.value_counts().sort_index().to_dict())

Training rows: 11577
Target counts: {0: 8683, 1: 2894}


## 3. Embedded method — tuned Decision Tree feature importance

The same best hyperparameters selected for the full-feature Decision Tree in Notebook 03 are reused here. This keeps the embedded ranking traceable to the model comparison rather than fitting an unrelated tree.

In [3]:
tree_row = model_summary.loc[
    (model_summary["model"] == "DecisionTree")
    & (model_summary["feature_set"] == "size_location_amenities")
]

if len(tree_row) != 1:
    raise ValueError("Expected exactly one full-feature DecisionTree row in model_summary.csv.")

best_params_raw = tree_row.iloc[0]["best_params"]
best_params = json.loads(best_params_raw)

# GridSearchCV stores pipeline parameter names such as model__max_depth.
tree_kwargs = {
    key.replace("model__", ""): value
    for key, value in best_params.items()
    if key.startswith("model__")
}
tree_kwargs["random_state"] = RANDOM_STATE

imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=FEATURES,
    index=X_train.index,
)

tree = DecisionTreeClassifier(**tree_kwargs)
tree.fit(X_train_imp, y_train)

embedded_scores = pd.Series(
    tree.feature_importances_,
    index=FEATURES,
    name="embedded_score",
).sort_values(ascending=False)

embedded_top3 = embedded_scores.head(3)
display(embedded_top3.to_frame())

,embedded_score
bedrooms,0.779258
distance_cbd_km,0.071486
bathrooms,0.059524


## 4. Filter method — Mutual Information

Mutual Information is calculated on the same training rows. Binary amenity indicators are declared discrete; the remaining numeric/count variables are treated as continuous. Median imputation is fitted only on training data.

In [4]:
DISCRETE_FEATURES = {
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_kitchen",
    "has_washer",
    "has_dryer",
}

discrete_mask = np.array([f in DISCRETE_FEATURES for f in FEATURES], dtype=bool)

mi_values = mutual_info_classif(
    X_train_imp,
    y_train,
    discrete_features=discrete_mask,
    random_state=RANDOM_STATE,
)

filter_scores = pd.Series(
    mi_values,
    index=FEATURES,
    name="filter_mi_score",
).sort_values(ascending=False)

filter_top3 = filter_scores.head(3)
display(filter_top3.to_frame())

,filter_mi_score
bedrooms,0.138877
accommodates,0.121966
bathrooms,0.107577


## 5. Compare the two top-3 lists

The comparison table preserves both ranks and scores. The report discussion should explain any disagreement using these actual values and the different mechanics of embedded vs filter selection.

In [5]:
rank_table = pd.DataFrame({
    "feature": FEATURES,
    "embedded_score": embedded_scores.reindex(FEATURES).values,
    "filter_mi_score": filter_scores.reindex(FEATURES).values,
})

rank_table["embedded_rank"] = rank_table["embedded_score"].rank(
    method="min", ascending=False
).astype(int)
rank_table["filter_rank"] = rank_table["filter_mi_score"].rank(
    method="min", ascending=False
).astype(int)
rank_table["rank_difference"] = (
    rank_table["embedded_rank"] - rank_table["filter_rank"]
).abs()

rank_table = rank_table.sort_values(
    ["embedded_rank", "filter_rank", "feature"]
).reset_index(drop=True)

display(rank_table)

top3_comparison = pd.DataFrame({
    "embedded_feature": embedded_top3.index.tolist(),
    "embedded_score": embedded_top3.values.tolist(),
    "filter_feature": filter_top3.index.tolist(),
    "filter_score": filter_top3.values.tolist(),
})
display(top3_comparison)

,feature,embedded_score,filter_mi_score,embedded_rank,filter_rank,rank_difference
0,bedrooms,0.779258,0.138877,1,1,0
1,distance_cbd_km,0.071486,0.022078,2,5,3
2,bathrooms,0.059524,0.107577,3,3,0
3,accommodates,0.059376,0.121966,4,2,2
4,amenity_count,0.013221,0.017422,5,6,1
5,has_pool,0.008128,0.000224,6,11,5
6,has_dryer,0.005246,0.000203,7,12,5
7,has_free_parking,0.001886,0.010682,8,7,1
8,beds,0.001875,0.104247,9,4,5
9,has_washer,0.000000,0.005074,10,8,2


,embedded_feature,embedded_score,filter_feature,filter_score
0,bedrooms,0.779258,bedrooms,0.138877
1,distance_cbd_km,0.071486,accommodates,0.121966
2,bathrooms,0.059524,bathrooms,0.107577


## 6. Hard-case listing

The rubric permits a listing that is a genuine outlier on the top-ranked feature. The procedure below is reproducible:

1. take the embedded method's top-ranked feature;
2. if it is numeric/non-binary, use the standard 1.5×IQR rule;
3. choose the outlier farthest from the feature median;
4. if no IQR outlier exists, choose the largest robust deviation and flag that fallback.

Actual values for all model features are saved for the selected listing.

In [6]:
top_feature = embedded_top3.index[0]

if top_feature in DISCRETE_FEATURES:
    # A binary feature cannot be a magnitude outlier. Fall back to the
    # highest-ranked non-binary embedded feature.
    non_binary_ranked = [f for f in embedded_scores.index if f not in DISCRETE_FEATURES]
    if not non_binary_ranked:
        raise ValueError("No non-binary feature available for hard-case outlier selection.")
    hard_feature = non_binary_ranked[0]
else:
    hard_feature = top_feature

x = pd.to_numeric(df[hard_feature], errors="coerce")
q1, q3 = x.quantile([0.25, 0.75])
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
median = x.median()

outlier_mask = x.lt(lower) | x.gt(upper)
candidate_idx = x.loc[outlier_mask].dropna().index
used_iqr_outlier = len(candidate_idx) > 0

if used_iqr_outlier:
    hard_idx = (x.loc[candidate_idx] - median).abs().idxmax()
else:
    valid_idx = x.dropna().index
    hard_idx = (x.loc[valid_idx] - median).abs().idxmax()

hard_cols = [ID_COL, "split", TARGET, "price_clean"] + FEATURES
hard_case = df.loc[[hard_idx], hard_cols].copy()
hard_case.insert(1, "hard_case_feature", hard_feature)
hard_case.insert(2, "used_1_5_iqr_outlier_rule", used_iqr_outlier)
hard_case.insert(3, "feature_q1", q1)
hard_case.insert(4, "feature_q3", q3)
hard_case.insert(5, "feature_iqr", iqr)
hard_case.insert(6, "lower_bound", lower)
hard_case.insert(7, "upper_bound", upper)
hard_case.insert(8, "feature_median", median)

display(hard_case.T)

,11724
id,1565029172672121402
hard_case_feature,bedrooms
used_1_5_iqr_outlier_rule,True
feature_q1,1.0
feature_q3,3.0
feature_iqr,2.0
lower_bound,-2.0
upper_bound,6.0
feature_median,2.0
split,train


## 7. Concrete opposite-ranking scenario

This table identifies the feature with the largest rank disagreement. It provides the actual group-specific feature needed to explain how a filter method can rank a feature highly while the embedded tree ranks it lower (or vice versa).

In [7]:
largest_disagreement = rank_table.sort_values(
    ["rank_difference", "filter_rank"],
    ascending=[False, True],
).iloc[[0]]

display(largest_disagreement)

,feature,embedded_score,filter_mi_score,embedded_rank,filter_rank,rank_difference
8,beds,0.001875,0.104247,9,4,5


## 8. Save reproducible feature-selection outputs

In [8]:
TABLE_OUT = REPO_ROOT / "output" / "tables"
TABLE_OUT.mkdir(parents=True, exist_ok=True)

rank_table.to_csv(TABLE_OUT / "feature_selection_rankings.csv", index=False)
top3_comparison.to_csv(TABLE_OUT / "feature_selection_top3.csv", index=False)
hard_case.to_csv(TABLE_OUT / "feature_selection_hard_case.csv", index=False)
largest_disagreement.to_csv(
    TABLE_OUT / "feature_selection_largest_rank_disagreement.csv",
    index=False,
)

print("Saved:")
for name in [
    "feature_selection_rankings.csv",
    "feature_selection_top3.csv",
    "feature_selection_hard_case.csv",
    "feature_selection_largest_rank_disagreement.csv",
]:
    print(" -", TABLE_OUT / name)

Saved:
 - /home/runner/work/COMP20008_A2_W04G10/COMP20008_A2_W04G10/output/tables/feature_selection_rankings.csv
 - /home/runner/work/COMP20008_A2_W04G10/COMP20008_A2_W04G10/output/tables/feature_selection_top3.csv
 - /home/runner/work/COMP20008_A2_W04G10/COMP20008_A2_W04G10/output/tables/feature_selection_hard_case.csv
 - /home/runner/work/COMP20008_A2_W04G10/COMP20008_A2_W04G10/output/tables/feature_selection_largest_rank_disagreement.csv


## 9. Evidence checklist

Before the group-written report is finalised, verify:
- embedded top 3 + scores are quoted correctly;
- filter top 3 + scores are quoted correctly;
- disagreement is explained using the actual rank table;
- the hard-case listing ID and relevant actual values are quoted;
- the opposite-ranking scenario names a real feature from this group's feature set;
- feature-selection limitations are specific to the observed results, not generic.